In [ ]:
import os, glob, traceback, multiprocessing, re
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from nltools.data import Brain_Data, Design_Matrix
from nilearn import image as nimg
from nltools.stats import regress, zscore
from nilearn.masking import compute_epi_mask

# Suppressing warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

In [ ]:
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

In [ ]:
base_dir   = os.path.abspath('/gpfs/gibbs/pi/levy_ifat/Nachshon/PSUB/altBIDS/derivatives/')
output_dir = '/gpfs/gibbs/pi/levy_ifat/Nachshon/PSUB/preProc_o'
os.makedirs(output_dir, exist_ok=True)

fwhm = 6
tr = 1
outlier_cutoff = 3

In [ ]:
file_list = [
    x for x in glob.glob(os.path.join(
        base_dir, 'sub-*', 'ses-*', 'func',
        'sub-207*_ses-1_task-trauma_run-*_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz'))
    if 'denoise' not in x
]
print(len(file_list))


In [ ]:
def load_run_with_matched_mask(f):
    img = nimg.load_img(f)
    # Prefer fMRIPrep's mask matching this exact run/space/res
    mask_candidates = glob.glob(f.replace('-preproc_bold.nii.gz', '-brain_mask.nii.gz'))
    mask_img = nimg.load_img(mask_candidates[0]) if mask_candidates else compute_epi_mask(img)
    mask_img = nimg.resample_to_img(mask_img, img, interpolation='nearest')  # align grid
    return Brain_Data(f, mask=mask_img)

def make_motion_covariates(mc, tr):
    z_mc = zscore(mc)
    all_mc = pd.concat([z_mc, z_mc**2, z_mc.diff(), z_mc.diff()**2], axis=1)
    all_mc.fillna(value=0, inplace=True)
    return Design_Matrix(all_mc, sampling_freq=1/tr)

def parse_bids_from_bold_path(f):
    """
    Require run-* in the BIDS basename.
    Example:
      sub-001_ses-1_task-traumarecall_run-2_space-..._desc-preproc_bold.nii.gz
    """
    bn = os.path.basename(f)
    m = re.search(
        r'^(sub-[^_]+)_ses-(?P<ses>[^_]+)_task-(?P<task>[^_]+?)_run-(?P<run>\d+)_space-',
        bn
    )
    if not m:
        raise ValueError(f'Expected run-* in filename: {bn}')
    sub  = m.group(1)
    ses  = m.group('ses')
    task = m.group('task')
    run  = m.group('run')  # guaranteed (string digits)
    return sub, ses, task, run

def confounds_path_for(f):
    """Exact confounds .tsv path matching sub/ses/task/run."""
    sub, ses, task, run = parse_bids_from_bold_path(f)
    func_dir = os.path.join(base_dir, sub, f'ses-{ses}', 'func')
    conf = os.path.join(
        func_dir,
        f"{sub}_ses-{ses}_task-{task}_run-{run}_desc-confounds_timeseries.tsv"
    )
    if not os.path.exists(conf):
        raise FileNotFoundError(f'Confounds not found: {conf}')
    return conf

def output_name_for(f, fwhm):
    """
    Preserve entities (incl. task/run/space/res) and replace desc-preproc with
    desc-preproc_denoise_smooth{fwhm}mm.
    """
    bn = os.path.basename(f)
    return bn.replace(
        'desc-preproc_bold.nii.gz',
        f'desc-preproc_denoise_smooth{fwhm}mm_bold.nii.gz'
    )

In [ ]:
def process_one(f):
    """Process a single BOLD *run*. Returns (sub, label, ok, msg)."""
    sub, ses, task, run = parse_bids_from_bold_path(f)
    label = f"{task}_run-{run}" if run is not None else task

    out_name = output_name_for(f, fwhm)
    out_path = os.path.join(output_dir, out_name)
    
    try:
        if os.path.exists(out_path):
            return sub, label, True, "SKIP: exists"
            
        conf_path = confounds_path_for(f)

        # load & smooth
        data = load_run_with_matched_mask(f)
        smoothed = data.smooth(fwhm=fwhm)

        # spikes
        spikes = smoothed.find_spikes(global_spike_cutoff=outlier_cutoff,
                                      diff_spike_cutoff=outlier_cutoff)

        # design matrix
        covariates = pd.read_csv(conf_path, sep='\t')
        # motion
        cols_mc = ['trans_x','trans_y','trans_z','rot_x','rot_y','rot_z']
        missing = [c for c in cols_mc if c not in covariates.columns]
        if missing:
            raise KeyError(f"Missing motion cols in confounds: {missing}")
        mc  = covariates[cols_mc]
        mc_cov = make_motion_covariates(mc, tr)

        # CSF (fallback to zeros if missing)
        if 'csf' in covariates.columns:
            csf = covariates[['csf']]
        else:
            csf = pd.DataFrame({'csf': np.zeros(len(covariates), dtype=float)})

        dm = Design_Matrix(pd.concat([csf, mc_cov, spikes.drop(columns='TR')], axis=1),
                           sampling_freq=1/tr)
        dm = dm.add_poly(order=2, include_lower=True)

        smoothed.X = dm
        stats = smoothed.regress()
        #
        stats['residual'].data = np.float32(stats['residual'].data)
        stats['residual'].write(out_path)

        return sub, label, True, 'OK'

    except Exception as e:
        return sub, label, False, f'{e.__class__.__name__}: {e}'

In [ ]:
# ---- parallel run (Jupyter-safe via joblib) ----
n_jobs = min(multiprocessing.cpu_count() or 1, 4)
print(f'Processing {len(file_list)} files with {n_jobs} workers')

results = Parallel(n_jobs=n_jobs, prefer="processes", verbose=0)(
    delayed(process_one)(f) for f in file_list
)

In [ ]:
# concise report (subject+run)
ok   = [(sub, label) for sub, label, success, _ in results if success]
fail = [(sub, label, msg) for sub, label, success, msg in results if not success]

for sub, label in ok:
    print(f'[OK]   {sub} :: {label}')

In [ ]:
for sub, label, msg in fail:
    print(f'[FAIL] {sub} :: {label} -> {msg}')

In [ ]:
print(f'\nDone. OK: {len(ok)} | FAIL: {len(fail)} | Unique subjects processed: {len(set([s for s,_ in ok]))}')

In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w -p xarray